In [1]:
!pip install pytesseract pdf2image pillow opencv-python anthropic
!apt-get install -q poppler-utils
!apt-get install -q tesseract-ocr-ara  # Arabic language pack

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 11.6 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (240 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be ins

مقدمة

نجدة فتحي صفوة

لا يعرف على وجه التحديد متى بدأ المغفور له جعفر العسكري بكتابة مذكراته، ولكن ذلك على أي حال، كان بعد عودته إلى العراق، واستقرار الحكم الوطني فيه، حيث تمكن من الخلود إلى شيء من الراحة، والشعور بشيء من الاستقرار بعد فترة طويلة من الأسفار والحروب في القصيم والعراق والبلقان وليبيا ومصر والحجاز وسورية .

... (Expected clean output after LLM correction)

In [2]:
import cv2
import numpy as np
import pytesseract
from pdf2image import convert_from_path
from PIL import Image

def optimize_pdf_ocr(pdf_path, lang='ara'):
    # 1. Convert PDF to list of PIL images at 300 DPI
    pages = convert_from_path(pdf_path, 300)

    full_text = []

    for i, page in enumerate(pages):
        # 2. Convert PIL image to OpenCV format (NumPy array)
        open_cv_image = np.array(page)

        # 3. Round-trip RGB->BGR->RGB (key step from Approach 3)
        image_rgb = cv2.cvtColor(open_cv_image, cv2.COLOR_RGB2BGR)
        image_rgb = cv2.cvtColor(image_rgb, cv2.COLOR_BGR2RGB)

        # 4. OCR
        text = pytesseract.image_to_string(image_rgb, lang=lang)
        full_text.append(f"--- Page {i+1} ---\n{text}")

    return "\n".join(full_text)

# Execution
pdf_file = 'pages_5_22.pdf'
optimized_result = optimize_pdf_ocr(pdf_file)

with open('optimized_approach.txt', 'w', encoding='utf-8') as f:
    f.write(optimized_result)

print("OCR Complete. Optimized text saved to optimized_approach.txt")

OCR Complete. Optimized text saved to optimized_approach.txt


In [3]:
import anthropic
from google.colab import userdata

# الإعدادات
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def refine_with_llm(raw_text, model_name='claude-haiku-4-5-20251001'):
    prompt = f"""
    أنت خبير في تحقيق النصوص التاريخية. المهمة هي معالجة نص OCR من مذكرات جعفر العسكري.

    المطلوب تنفيذ القواعد التالية بدقة:
    1. دمج الأسطر (Flowing Text): قم بإلغاء الفواصل المصطنعة بين الأسطر الناتجة عن التنسيق العمودي للكتاب، واجعل النص يتدفق كفقرات متصلة وطبيعية.
    2. إصلاح الأخطاء فقط: قم بتصحيح الأخطاء الإملائية الناتجة عن القراءة الآلية (مثلاً: "نباية مؤلة" تصبح "نهاية مؤلمة"، "القضيع والسراق" تصبح "القصيم والعراق") دون تغيير الأسلوب أو إضافة كلمات خارجية.
    3. سياق الكتاب: حافظ على الأسماء التاريخية والجغرافية كما وردت (البلقان، الحجاز، سورية).
    4. الاستمرارية: لا تضع نقطة نهاية في آخر النص إذا كان الكلام ينتهي بجملة مفتوحة (لأنها تستكمل في الصفحة التالية).
    5. التنسيق: حافظ على العناوين الرئيسية (مثل اسم المؤلف أو كلمة مقدمة) في أسطر مستقلة.
    6. أخرِج النص المصحح مباشرةً فقط، بدون أي عنوان أو تعليق في البداية.

    النص الخام المطلوب معالجته:
    {raw_text}
    """

    try:
        message = client.messages.create(
            model=model_name,
            max_tokens=4096,
            messages=[{"role": "user", "content": prompt}]
        )
        return message.content[0].text
    except Exception as e:
        return f"Error: {e}"

# التنفيذ
raw_ocr = optimized_result  # النتيجة من الكود السابق
clean_text = refine_with_llm(raw_ocr)

print("\n--- النص المرمم بواسطة الذكاء الاصطناعي ---\n")
print(clean_text)


--- النص المرمم بواسطة الذكاء الاصطناعي ---

نجدة فتحى صفوة

لا يعرف على وجه التحديد متى بدأ المغفور له جعفر العسكري بكتابة مذكراته. ولكن ذلك، على أي حال، كان بعد عودته إلى العراق واستقرار الحكم الوطني فيه، حيث تمكن من الخلود إلى شيء من الراحة والشعور بشيء من الاستقرار بعد فترة طويلة من الأسفار والحروب في القصيم والعراق والبلقان وليبيا ومصر والحجاز وسورية.

إن حياة جعفر العسكري وما شهده من أحداث حاسمة وخطيرة في تاريخ العراق والأمة العربية تجعل مذكراته ذات قيمة تاريخية كبيرة. وسيرة جعفر العسكري نفسها حافلة بالأحداث والمخاطر التي رمى بنفسه إليها تحقيقاً لفكرة وإيماناً بمبدأ. وهي لصح أن تكون مادة لرواية ذات دروس وعبر، وقد انتهت بنهاية مؤلمة حقاً، ولكنها مع ذلك قد تتسق بدرجة غريبة مع سيرته التي كانت كلها عبارة عن أحداث مثيرة ومغامرات جريئة ومفاجآت. ولو شاء كاتب قصصي أن يكتب رواية عن شخصية خيالية مرّ بها ما مرّ بجعفر العسكري من أحداث، لا استطاع أن يجد لقصته نهاية تتسق مع بدايتها، ولا خاتمة تكون أبلغ دلالة، ولا أروع إثارة، ولا أكثر إيلاماً من الخاتمة التي انتهت بها حياة ذلك الرجل العظيم.

و

In [4]:
# Save the corrected text
output_filename = 'LLM_corrected_text.txt'
with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(clean_text)

print(f'Corrected Arabic text saved to {output_filename}')

Corrected Arabic text saved to LLM_corrected_text.txt
